# Agente de inversión con Q-learning y DQN

Cuaderno de entrada al proyecto. El código del entorno y de los agentes está en src. Las semillas y los parámetros están en configs/default.json. Ejecuta primero eda.ipynb para inspeccionar entrenamiento.

In [ ]:
from pathlib import Path
import os, sys, json
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
assert (root / "configs/default.json").exists(), "Abre el cuaderno desde el proyecto"
os.chdir(root)
sys.path.insert(0, str(root))
os.environ.setdefault("MPLCONFIGDIR", "/tmp/rl_inversion_matplotlib")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.data import prepare, FEATURES
cfg = json.loads(Path("configs/default.json").read_text())
df, x, splits, mu, sigma = prepare(cfg["data"], cfg)
train = df.iloc[:splits["train"][1]+1].copy()
print("EDA restringido a entrenamiento:", train.timestamp.iloc[0], "a", train.timestamp.iloc[-1])

## Comprobar la contabilidad con un paso

Las acciones 0, 1 y 2 representan pesos objetivo de 0 %, 50 % y 100 % después de pagar comisiones.

In [ ]:
from src.environment import InvestmentEnv
from stable_baselines3.common.env_checker import check_env
env = InvestmentEnv(df, x, *splits["train"], initial_capital=cfg["initial_capital"], cost_rate=cfg["cost_rate"])
check_env(env)
obs, info = env.reset(seed=7)
display(pd.Series(obs, index=FEATURES+["peso_actual", "drawdown"]))
obs, reward, terminated, truncated, info = env.step(2)
display(pd.Series(info))

## Entrenar y evaluar

La siguiente celda crea una réplica completa solo si aún no existe. Selecciona algoritmo y recompensa con validación antes de recorrer prueba. No cambies parámetros después de leer el resultado final si deseas mantener una evaluación intacta.

In [ ]:
import subprocess
run = Path("results/desde_notebook")
if not run.exists():
    subprocess.run([sys.executable, "-m", "src.train", "--config", "configs/default.json", "--output", str(run)], check=True)
else:
    print("Se conservan los resultados existentes en", run)
assert (run/"test_metrics.csv").exists(), "La ejecución no terminó; usa otra carpeta tras resolver el error"

## Resultados reproducibles

La selección se lee del archivo congelado, no se cambia por lo ocurrido en prueba.

In [ ]:
display(json.loads((run/"selection_frozen.json").read_text()))
display(pd.read_csv(run/"validation_metrics.csv"))
display(pd.read_csv(run/"test_metrics.csv"))

In [ ]:
from IPython.display import Image, Markdown
for image in ["equity_drawdown.png", "positions.png", "reward_comparison.png", "learning_curves.png"]:
    display(Image(filename=str(run/image)))
display(Markdown((run/"INFORME.md").read_text().replace("](", "](../"+str(run)+"/")))

## Reflexión y entrega

Revisa el peor paso de cada semilla, compara las comisiones y comprueba si el agente permanece fuera del mercado. El informe contiene una guía para la reflexión personal. GUION_EXPOSICION.md organiza una presentación de cinco minutos. No se afirma que una sola ventana pruebe rentabilidad futura.